# Lösung der allgemeinen Legendre Differentialgleichung

In [ ]:
# Imports für alle Vorlesungsnotebooks
import sympy as sp

# Pfad für sympy_utils Modul hinzufügen
import sys

sys.path.append("..")

import sum_utils as su
import power_series_tools as pst

In diesem Notebook möchte ich die Schritte zur Lösung der allgemeinen Legendre Differentialgleichung vorstellen. Sie entsteht unter anderem als Winkelanteil der dreidimensionalen Laplace-Gleichung in Kugelkoordinaten und ihre Lösungen sind zentraler Bestandteil der Kugelflächenfunktionen. 

Die DGL für $y(x)$ lautet:
$$
(1 - x^2) \frac{d^2 y}{dx^2} - 2x \frac{dy}{dx} + \left[ \ell (\ell + 1) - \frac{m^2}{1 - x^2} \right] y = 0 \tag{1}
$$

und wir suchen *reguläre* Lösungen für $x\in[-1, 1]$.

Die allgemeine Legendre DGL ist eine Verallgemeinerung der Legendre DGL mit $m=0$:

$$
(1 - x^2) \frac{d^2 y}{dx^2} - 2x \frac{dy}{dx} + \ell (\ell + 1) y = 0 \tag{2}
$$

Die Lösungen von (1) lassen sich aus denen von (2) gewinnen. Daher werden wir auch die Lösung von (2) bestimmen.

## Implementation der Gleichungen in SymPy

In [ ]:
# Definiere die Symbole
x = sp.symbols('x')
l, m, n = sp.symbols('l, m, n', integer=True, positive=True)

rho = sp.symbols(r'\rho', real=True, positive=True)
a = sp.symbols('a', cls=sp.Function)

# Die Funktionen y und deren Ableitungen
y = sp.Function('y')(x)
y_d = y.diff(x)
y_dd = y.diff(x, x)

# Symbole für die Polynomlösungen:
Plm = sp.Function(r'P_l^m')(x)
Plmp1 = sp.Function(r'P_l^{m+1}')(x)
P = sp.Function(r'P')(x)


# 1. Klassische Legendre-Differentialgleichung
lhs_classic = (1 - x**2) * y_dd - 2*x * y_d + l * (l + 1) * y
classic_eq = sp.Eq(lhs_classic, 0)

# 2. Verallgemeinerte Legendre-Differentialgleichung
lhs_generalized = (1 - x**2) * y_dd - 2*x * y_d + (l * (l + 1) - m**2 / (1 - x**2)) * y
generalized_eq = sp.Eq(lhs_generalized, 0)

# Ausgabe der beiden Gleichungen
print("Klassische Legendre-Differentialgleichung (m=0):")
display(classic_eq)

print("Verallgemeinerte Legendre-Differentialgleichung:")
display(generalized_eq)


Im folgenden beschreibe ich die Schritte/Gedanken, die zu den Lösungen von (1) führen, welche in der Physik verwendet werden.

### 1. Reguläre Lösungen

Einer der wesentlichen Vorraussetzungen bei der Lösung von (1) für uns ist, dass wir für physikalische Anwendungen *reguläre* Lösungen für $x\in[-1, 1]$ finden wollen.

Wir bemerken jedoch, dass die Differentialgleichung für $x=\pm 1$ singulär ist. Um trotzdem Lösungen zu finden, die bei $x=\pm1$ regulär sind, wird zunächst ein Ansatz der Form

$$
y(x) = (1 - x^2)^{\rho} P(x) \tag{3}
$$

probiert, wobei $P(x)$ eine zunächst unbekannte, aber auf $x\in[-1, +1]$ reguläre Funktion sein soll. Wir müssen testen, ob ein $\rho > 0$ existiert, so dass (3) für alle $x\in[-1, +1]$ eine reguläre Lösung von (1) ergibt.

In [ ]:
# Einsetzen des Ansatzes in die DGL:
ansatz = (1 - x**2)**rho * P
display(sp.Eq(y, ansatz))

eq_1 = generalized_eq.subs({y : ansatz})
display(eq_1)

In [ ]:
# Ableitungen ausführen und den Ausdruck (linke Seite der Gleichung) umformen:
eq_2 = sp.use(eq_1.doit().expand().lhs.collect((1-x**2)**rho), sp.collect, args=(P,), level=1)
eq_2

Wenn möglich, wollen wir $\rho$ so festlegen, dass in obigem Ausdruck bei $x=\pm 1$ keine Singularität mehr auftritt. Wir haben potentiell singuläre Terme bei $P'(x)$ und $P(x)$. Für den Term bei $P'(x)$ ist aber klar, dass er für alle $\rho$ im Grenzwert $x\to (-1, 1)$ keine Probleme macht:

In [ ]:
# isoliere relevante Terme
eq_2_tmp = (eq_2 / (1-x**2)**rho)

coeff_P = eq_2_tmp.coeff(P)
coeff_P_d = eq_2_tmp.coeff(sp.Derivative(P, x))

display(coeff_P)
display(coeff_P_d)

In [ ]:
# behandle den Term P'(x):
limit_1 = sp.Limit(coeff_P_d, x, -1)
limit_2 = sp.Limit(coeff_P_d, x, 1, dir='-')
display(sp.Eq(limit_1, limit_1.doit()))
display(sp.Eq(limit_2, limit_2.doit()))

Für den Term von $P(x)$ ist der Ausdruck und die Analyse etwas komplexer:

In [ ]:
# behandle den Term P(x):
limit_1 = sp.Limit(coeff_P.simplify(), x, -1)
display(sp.Eq(limit_1, limit_1.doit()))

Wir sehen, dass wir überhaupt nur eine Chance auf einen nicht unendlichen Grenzwert haben, wenn
$\rho=\pm \frac{m}{2}$. Wir probieren den Wert $\rho=\frac{m}{2}$ für die relevanten Grenzwerte. Die Möglichkeit $\rho=-\frac{m}{2}$ verfolgen wir nicht weiter, da dies automatisch zu einer irregulären Lösung der DGL führen würde. 

In [ ]:
expr_rho = coeff_P.simplify().subs({rho : m / 2})

limit_1 = sp.Limit(expr_rho, x, -1)
display(sp.Eq(limit_1, limit_1.doit()))

limit_2 = sp.Limit(expr_rho, x, 1, dir='-')
display(sp.Eq(limit_2, limit_2.doit()))

Wir sehen, dass wir mit dem Ansatz

$$
y(x) = (1 - x^2)^{m/2} P(x)
$$

reguläre Lösungen unserer DGL erwarten dürfen.

### 2. Lösungen der Form $y(x) = (1 - x^2)^{m/2} P(x)$

Wir machen weiter, indem wir mit diesem Ansatz eine DGL für $P(x)=P_l^m(x)$ gewinnen. Unsere Lösung hängt sicher von den Parametern $l$ und $m$ ab.

In [ ]:
# Mit dem Ansatz in die DGL:
ansatz = (1 - x**2)**(m/2) * Plm

eq_3 = generalized_eq.subs({y : ansatz})
display(eq_3)

In [ ]:
# Linke Seite umformen und vereinfachen:
eq_3_1 = eq_3.lhs.doit().simplify().collect(Plm) / (1-x**2)**(m/2)
eq_3_1

Dies ist eine Differentialgleichung für die regulären Funktionen $P_l^m(x)$.

An dieser Stelle muss man erkennen oder wissen, dass man durch Ableiten dieser Gleichung für $P_l^m$ nach $x$ die Differentialgleichung für $P_l^{m+1}$ erhalten kann:

In [ ]:
# Gleichung ableiten und umformen:

eq_3_2 = sp.diff(eq_3_1, x)
display(eq_3_2)
eq_3_3 = eq_3_2.collect(Plm)
display(eq_3_3)
eq_3_4 = eq_3_3.subs(sp.Derivative(Plm, x), Plmp1)
display(eq_3_4)

In [ ]:
# Vergleich mit obiger Gleichung: m->(m+1) und P_l^m -> P_l^(m+1):
eq_3_5 = eq_3_1.subs({m : m + 1, Plm : Plmp1})
display(eq_3_5)

sp.simplify(eq_3_4 - eq_3_5)

Wir folgern, dass wir mit unserem Ansatz *nur* Lösungen der Gleichung für $m=0$ (die Legendre DGL) finden müssen und dann Lösungen für (1) durch:

$$
P_l^m = (1 - x^2)^{m/2}\frac{d^m}{dx^m} \underbrace{P_l^0(x)}_{=:P_l(x)}
$$

gewinnen können.

**Bemerkungen:** 

1. Obiges macht implizit die Annahme, dass $m$ nur ganzzahlige Werte annimmt. An dieser Stelle wissen wir aus der Differentialgleichung aber nicht, ob es nicht noch andere Lösungen mit rellen $m$ gibt. Im allgemeinen existieren in der Tat Lösungen von (1) für beliebige reelle oder komplexe Werte von $l$ und $m$. Diese Lösungen sind aber bei $x=\pm 1$ nicht regulär, wenn $l$ und $m$ nicht ganzzahlig sind!

2. In der Elektrodynamik kommen wir im Rahmen des Laplace-Operators in Kugelkoordinaten auf (1). Dort sind für $m$, welches aus dem Winkel $\phi$-Anteil des Laplace-Operators stammt, von vorneherein nur ganzzahlige Werte zulässig. Die Funktion $e^{im\phi}$ muss aus physikalischen Gründen für volle Umdrehungen von $\phi$ denselben Wert besitzen!

3. Aus der folgenden Analyse von (2) werden wir sehen, dass aufgrund der Regularitätsforderung $l$ eine natürliche Zahl sein muss. Dies wird uns auch Grenzen für $m$ geben.

### 3. Die Lösung der Legendre Differentialgleichung

Im folgenden bestimmen wir Lösungen $P_l(x)$ von (2), welche aus Gleichung (1) durch $m=0$ hervorgeht:

$$
(1 - x^2) \frac{d^2 P_l(x)}{dx^2} - 2x \frac{dP_l(x)}{dx} + \ell (\ell + 1) P_l(x) = 0 \tag{2}
$$


Standardverfahren zur analytischen Lösung von DGL (Separation der Variablen, Substitutionen etc.) führen hier nicht zum Ziel. In solchen Fällen können wir versuchen, Lösungen durch einen Potenzreihenansatz zu finden:

$$
P_l(x)=\sum_{n=0}^{\infty}a_nx^n.
$$

In [ ]:
ansatz = sp.Sum(a(n) * x**n, (n, 0, sp.oo))
ansatz

In [ ]:
# Den Ansatz in die DGL einsetzen und umformen. Beachte die Verwendung der
# eigenen Funktion diff_pow_series, welche eine geeignete Normierung
# der Indizes für Summenableitungen durchführt:
expr = classic_eq.subs({sp.Derivative(y, x, 2) : pst.diff_power_series(ansatz, x, 2),
                        sp.Derivative(y, x) : pst.diff_power_series(ansatz, x),
                        y : ansatz}, simultaneous=True)
expr

Ziel der folgenden Zellen ist es, die Summen über eine Potenz $x^n$ in einer großen Summe zusammenzufassen:

In [ ]:
# Summen zusammenfassen:

# eigene Klasse/Funktion, um Summen zusammenzufassen und Exponenten zu
# normieren:
norm = pst.PowerSumNormalizer(x, n) # eigene Klasse/Funktion
res = norm.normalize(su.push_prefactors_into_sums(expr.lhs).simplify().expand())
expr1 = su.push_prefactors_into_sums(res.simplify())
sp.Eq(expr1, 0)

Es fällt auf, dass wir die letzten beiden Summen auch von $n=0$ loslaufen lassen können und daher alles unter einer Summe vereinigen können. Für das folgende brauchen wir nur den Summanden dieser Summe:

In [ ]:
# Normalisiere untere Indizes aller Summen:
expr2 = su.rewrite_sum_lower_limit(expr1, 0)
display(expr2)

# fasse die Summen zusammen:
expr3 = su.collect_operator_terms(expr2, sp.Sum, ((n, 0, sp.oo), ))
display(expr3)

# und extrahiere den Summanden:
summand = expr3.function
display(summand)

In [ ]:
# Da die x^n in der Summe alle unabhängig voneinander sind, muss dieser Ausdruck
# identisch verschwinden, was uns eine Rekursionsformal für die an liefert:
expr1 = sp.solve(summand, a(n+2))[0].factor()
sp.Eq(a(n+2), expr1)

**Anmerkungen:**

- $a_0$ und $a_1$ sind frei wählbar (Anfangsbedingungen).
- Da die Rekursionsformel nur den $n+2$-ten mit dem $n$-ten Term der Potenzreihe verknüpft, zerfällt die Potenzreihe in zwei *unabhängige* Reihen. Dies sind die zwei unabhängigen Lösungen von (2) für gegebenenes $l$.
- Für ganzzahlige $l$ bricht eine der Lösungen bei $n=l$ ab und wir erhalten daraus die (Legendre)polynome. Die zweite Lösung konvergiert nur für $|x| < 1$ und ist daher für uns nicht interessant.

Schauen wir uns die Lösung konkret für einige $l$ an:

In [ ]:
# Die Funktion RecursiveSeq erlaubt das konkrete berechnen von Termen
# einer Rekursionsformel:
from sympy.series.sequences import RecursiveSeq

a0, a1 = sp.symbols(r'a_0, a_1')

# Die Rekursionsformel muss in der Form a(n) = .... formuliert werden. Wir
# müssen daher den n-Index um 2 verschieben:
a_rec = RecursiveSeq(expr1.subs({n : n - 2}), a(n), n, [a0, a1])
a_rec.recurrence

In [ ]:
l_val = 3
n_terme = 50

a_vals = a_rec[:n_terme]
monom = [x**i for i in range(n_terme) ]
monom

S = sp.Add(*[ m * e for m, e in zip(monom, a_vals)])
S.collect([a0, a1]).subs({l : l_val})

Wir sehen, dass abwechselnd für $l$ die geraden bzw. ungeraden Reihen abbrechen und die Legendre Polynome liefern. Mit den Konstanten $a_0$ und $a_1$ ist noch die Normierung $P_l(1)=1$ sicherzustellen.